<a href="https://colab.research.google.com/github/parika8ec-hub/DA_AI_Project2_Stock_Data/blob/main/Project2_AdvancedEDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task-1: Advanced Data Cleaning

In [2]:
#Import Libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
#Load cleaned dataset of Project1
cleaned_data = pd.read_csv('merged_stock_data.csv', parse_dates=['date'], index_col='date')
#Display few rows of dataset
print('Few rows of dataset:')
print('-'*75)
print(cleaned_data.head())

#Display information of dataset
print('\nDataset information:')
print('-'*45)
print(cleaned_data.info())

Few rows of dataset:
---------------------------------------------------------------------------
           ticker   open  close  adj_close    low   high     volume  \
date                                                                  
2013-05-08    AHH  11.50  11.58   8.493155  11.25  11.68  4633900.0   
2013-05-09    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0   
2013-05-10    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0   
2013-05-13    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0   
2013-05-14    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0   

            price_decade   sector  
date                               
2013-05-08        2010.0  FINANCE  
2013-05-09        2010.0  FINANCE  
2013-05-10        2010.0  FINANCE  
2013-05-13        2010.0  FINANCE  
2013-05-14        2010.0  FINANCE  

Dataset information:
---------------------------------------------
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 513243 entries, 2013-05-08 to 20

## Handle Missing Values:

***Purpose:***

The purpose of handling missing values was to improve data quality and integrity by ensuring the dataset was complete and suitable for analysis and predictive modeling. Missing values in stock market datasets can lead to inaccurate trends, biased model predictions and unreliable insights. Therefore, appropriate imputation techniques were applied based on the nature of each variable.

***Justification on Techniques:***

- The missing value handling techniques were chosen based on the characteristics of each variable to ensure accurate imputation while preserving data integrity.
- Missing values in the sector column were replaced with "Unknown" because it is a categorical variable and missing sector information cannot be estimated numerically; this approach prevents unnecessary data loss while preserving records.
- For open, close, and adjusted close prices, forward fill and backward fill methods were applied within each ticker because these are critical stock prices recorded daily and follow a chronological time sequence. Forward fill uses previous known values, while backward fill addresses missing values at the beginning of a stock's trading history, helping maintain time-series continuity without introducing unrealistic price fluctuations.
- Missing values in high and low prices were handled using linear interpolation because these variables represent intraday price ranges and interpolation estimates missing values using surrounding observations, making it suitable for filling small gaps in continuous numerical data while preserving stock price trends.
- For the volume column, median imputation was used because trading volume often contains extreme values or outliers and the median is more robust than the mean, providing stable replacement values without being affected by large spikes.

***Used Methodology:***

The missing value handling process followed a variable-specific imputation methodology, where different techniques were applied depending on the type and behavior of the data.

- Categorical variables used constant replacement as Unknown
- Time-series price variables used forward fill + backward fill
- Continuous range variables used linear interpolation
- Skewed numerical variables used median imputation

This hybrid approach ensured that missing values were handled appropriately while maintaining the financial dataset's temporal structure and statistical reliability.

In [4]:
#Check missing values
print('\nMissing values:')
print('-'*45)
print(cleaned_data.isnull().sum())

#Filling missing values of categorical column by Unknown
cleaned_data['sector'] = cleaned_data['sector'].fillna("Unknown")

#Filling 'open','close' and 'adj_close' columns using forward fill(ffill)/backward fill(bfill) within each ticker to maintain time-series consistency
price_cols1 = ['open', 'close', 'adj_close']#take numerical columns
cleaned_data[price_cols1] = cleaned_data.groupby('ticker')[price_cols1].transform(lambda x: x.ffill().bfill())

#Fill missing values in high and low columns using linear interpolation within each ticker group to estimate intermediate values based on time trends
# while preserving the sequential nature of stock price movements
price_cols2 = ['high', 'low']#take numerical columns
cleaned_data[price_cols2] = cleaned_data.groupby('ticker')[price_cols2].transform(lambda x: x.interpolate(method='linear'))

#Fill missing volume values within each ticker using the median volume to reduce the impact of extreme values and maintain robustness in trading activity data
cleaned_data['volume'] = cleaned_data.groupby('ticker')['volume'].transform(lambda x: x.fillna(x.median()))

#Verify updated missing values
print('\nUpdated Missing values:')
print('-'*45)
print(cleaned_data.isnull().sum())


Missing values:
---------------------------------------------
ticker          0
open            0
close           0
adj_close       0
low             0
high            0
volume          0
price_decade    1
sector          1
dtype: int64

Updated Missing values:
---------------------------------------------
ticker          0
open            0
close           0
adj_close       0
low             0
high            0
volume          0
price_decade    1
sector          0
dtype: int64


## Detect and Resolve Outliers:

***Purpose:***

The purpose of outlier detection and treatment was to identify extreme values in critical numerical variables such as volume and close prices that could negatively impact statistical analysis and predictive modeling. Outliers can distort distributions, influence model performance and lead to unreliable insights. Therefore, detecting and managing these extreme values helps improve data quality and ensures more stable analytical results.

***Justification on Technique:***

The IQR (Interquartile Range) method was selected because it is a robust statistical technique that works well for financial datasets and does not assume a normal distribution. Stock market data, particularly trading volume and closing prices, often contains extreme values due to sudden market fluctuations, unusual trading activity, or data recording errors. Instead of removing these observations, a capping (winsorization) approach was used because some extreme values may represent genuine market behavior. Capping limits the influence of extreme observations while preserving important records and maintaining the overall dataset size.

***Used Methodology:***

Outliers in the volume and close columns were detected using the IQR method by calculating the first quartile (Q1), third quartile (Q3) and the interquartile range (IQR = Q3 - Q1). Lower and upper bounds were then determined using the formula:

Lower Bound = Q1 - 1.5 x IQR

Upper Bound = Q3 + 1.5 x IQR

Any values falling outside these bounds were identified as outliers and counted to assess their frequency.

After detection, extreme values in both columns were handled using clipping, where values below the lower bound were replaced with the lower limit and values above the upper bound were replaced with the upper limit. This ensured that extreme values were controlled without removing potentially valuable market observations.

In [5]:
#Detect outliers in volume column using IQR method
Q1 = cleaned_data['volume'].quantile(0.25)#Compute the first quartile (Q1) to calculate IQR
Q3 = cleaned_data['volume'].quantile(0.75)#Compute the third quartile (Q3) to calculate IQR

IQR = Q3 - Q1#Compute IQR

# Define lower and upper bounds (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Count outliers before handling
volume_outliers = cleaned_data[(cleaned_data['volume'] < lower) | (cleaned_data['volume'] > upper)]
print("Number of volume outliers:", len(volume_outliers))

# Handle outliers by capping extreme volume values within the IQR-based lower and upper bounds using clipping
cleaned_data['volume'] = cleaned_data['volume'].clip(lower, upper)
#-----------------------------------------------------------------------------------------
# Detect outliers in close column using IQR method
Q1 = cleaned_data['close'].quantile(0.25)#Compute the first quartile (Q1) to calculate IQR
Q3 = cleaned_data['close'].quantile(0.75)#Compute the third quartile (Q3) to calculate IQR

IQR = Q3 - Q1#Compute IQR

# Define lower and upper bounds (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Count outliers before handling
volume_outliers = cleaned_data[(cleaned_data['close'] < lower) | (cleaned_data['close'] > upper)]
print("Number of close outliers:", len(volume_outliers))

# Handle outliers by capping extreme close values within the IQR-based lower and upper bounds using clipping
cleaned_data['close'] = cleaned_data['close'].clip(lower, upper)

Number of volume outliers: 72389
Number of close outliers: 41589


## Error Identification and Correction:

***Purpose:***

The purpose of error identification and correction was to ensure data integrity, accuracy and consistency before performing analysis and predictive modeling. Errors such as logical inconsistencies in stock prices, negative values and duplicate records can distort analysis results and reduce model reliability. Detecting and correcting these issues helps maintain a clean dataset that accurately reflects real market behavior.

***Justification on Technique:***

These validation techniques were chosen because stock market data follows strict logical relationships that must be maintained. For example, the high price should always be greater than or equal to open, close and low prices, while the low price should always be less than or equal to open, close and high prices. Any violations indicate potential data entry or collection errors. Similarly, negative stock prices or trading volumes are unrealistic in financial datasets and must be identified. Duplicate records were also checked because repeated observations can bias analysis and lead to inaccurate results. These techniques ensure that only valid and reliable records remain in the dataset.

***Used Methodology:***

- A rule-based validation approach was used to identify data inconsistencies.
- First, logical checks were applied to detect rows where the high price was lower than open, close and low prices, and where the low price was higher than open, close and high prices.
- Next, numerical validation checks were performed to identify any negative values in stock price and volume columns.
- Finally, duplicate records were detected using duplication checks.
- After identifying these errors, invalid rows and duplicate records were removed from the dataset to maintain consistency and improve overall data quality.

In [6]:
# Detect rows where the high price is lower than open, close and low prices
invalid_high = cleaned_data[cleaned_data['high'] < cleaned_data[['open', 'close', 'low']].max(axis=1)]

# Detect rows where the low price is higher than open, close and high prices
invalid_low = cleaned_data[cleaned_data['low'] > cleaned_data[['open', 'close', 'high']].min(axis=1)]

# Display the number of invalid high price records
print("Invalid high rows:", len(invalid_high))

# Display the number of invalid low price records
print("Invalid low rows:", len(invalid_low))


# Detect rows containing negative values in price and volume columns
# Negative stock prices or trading volume are considered invalid
invalid_values = cleaned_data[(cleaned_data[['open', 'close', 'high', 'low', 'volume']] < 0).any(axis=1)]

# Display the number of rows with negative values
print("Negative value rows:", len(invalid_values))

# Identify duplicate rows in the dataset
duplicates = cleaned_data.duplicated().sum()

# Display the total number of duplicate records
print("Duplicate rows:", duplicates)

Invalid high rows: 206
Invalid low rows: 40865
Negative value rows: 0
Duplicate rows: 2885


In [7]:
# Remove duplicate rows from the dataset to avoid repeated records
cleaned_data = cleaned_data.drop_duplicates()

# Verify that all duplicate rows have been removed
print("Duplicate rows count:", cleaned_data.duplicated().sum())

# Remove rows containing negative values in price and volume columns
cleaned_data = cleaned_data[(cleaned_data[['open', 'close', 'high', 'low', 'volume']] >= 0).all(axis=1)]


# Remove rows with logical inconsistencies in stock prices as keep rows where high price is greater than or equal to open, close and low prices
# and keep rows where low price is less than or equal to open, close and high prices
cleaned_data = cleaned_data[
    (cleaned_data['high'] >= cleaned_data[['open', 'close', 'low']].max(axis=1)) &
    (cleaned_data['low'] <= cleaned_data[['open', 'close', 'high']].min(axis=1))]

# Display the updated dataset shape after all corrections
print('Updated cleaned data shape:', cleaned_data.shape)

Duplicate rows count: 0
Updated cleaned data shape: (469332, 9)


# Task-2: Data Transformation

## Feature Engineering:

***Purpose:***

The purpose of feature engineering was to transform raw stock market data into meaningful predictive variables that capture important market behaviors such as trends, volatility and price momentum. These engineered features enhance the dataset's ability to support machine learning models by providing deeper insights into stock price movements over time.


***Justification on Techniques:***

Stock market data is highly time-dependent and influenced by both short-term fluctuations and long-term trends. Simple raw variables such as open and close prices do not fully capture market dynamics. Therefore, additional features were created to better represent underlying patterns. Moving averages help smooth price fluctuations and identify trends, volatility measures capture the level of risk or instability in stock prices and daily returns reflect momentum and short-term price changes. These features improve the predictive power of the dataset and make it more suitable for financial forecasting models.

***Used Methodology:***

Feature engineering was performed using a group-wise transformation approach based on each stock ticker to preserve time-series structure.
- A 7-day and 30-day moving average of closing prices were calculated to capture short-term and long-term trends respectively using rolling mean functions.
- A 7-day rolling standard deviation was computed to measure price volatility, indicating how much the closing price fluctuates over a short period.
- Additionally, daily returns were calculated using percentage change in closing prices to represent day-to-day market movement.

All transformations were applied using grouped operations to ensure that calculations were performed independently for each stock and maintain temporal consistency.

In [8]:
#Use Rolling Averages:

# Create a 7-day moving average feature of closing prices for each ticker and this captures short-term price trends by averaging the last 7 trading days
cleaned_data['close_ma7'] = cleaned_data.groupby('ticker')['close'].transform(lambda x: x.rolling(window=7).mean())

# Create a 30-day moving average feature of closing prices for each ticker and this captures long-term price trends by averaging the last 30 trading days
cleaned_data['close_ma30'] = cleaned_data.groupby('ticker')['close'].transform(lambda x: x.rolling(window=30).mean())

In [9]:
#Measure Volatility:

# Create a 7-day rolling standard deviation feature of closing prices for each ticker and this measures short-term price volatility by
#calculating how much closing prices fluctuate over the past 7 trading days
cleaned_data['close_volatility'] = cleaned_data.groupby('ticker')['close'].transform(lambda x: x.rolling(window=7).std())

In [10]:
#Compute Daily Returns:

# Compute daily returns for each ticker based on percentage change in closing price and this captures day-to-day price movement and is useful for
# momentum and prediction analysis
cleaned_data['daily_return'] = cleaned_data.groupby('ticker')['close'].pct_change()

# Display first few rows of the updated dataset to verify new feature creation
print('Few rows of updated dataset:')
print(cleaned_data.head())

Few rows of updated dataset:
           ticker   open  close  adj_close    low   high     volume  \
date                                                                  
2013-05-08    AHH  11.50  11.58   8.493155  11.25  11.68  1223300.0   
2013-05-09    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0   
2013-05-10    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0   
2013-05-13    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0   
2013-05-14    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0   

            price_decade   sector  close_ma7  close_ma30  close_volatility  \
date                                                                         
2013-05-08        2010.0  FINANCE        NaN         NaN               NaN   
2013-05-09        2010.0  FINANCE        NaN         NaN               NaN   
2013-05-10        2010.0  FINANCE        NaN         NaN               NaN   
2013-05-13        2010.0  FINANCE        NaN         NaN               NaN   
2013-

## Data Normalization/Standardization:

***Purpose:***

The purpose of data normalization was to ensure that all numerical features are brought to a similar scale so that no single variable dominates the model due to differences in magnitude. This improves the performance and stability of machine learning algorithms by making the dataset more suitable for training.

***Justification on Techniques:***

The stock market dataset contains numerical variables such as open, close, high, low and volume, which operate on different scales. For example, volume values are typically much larger than price values. Without scaling, machine learning models may become biased toward features with larger numerical ranges. Standardization using StandardScaler was chosen because it transforms the data to have a mean of 0 and a standard deviation of 1, making it appropriate for models that assume normally distributed or scaled inputs.

***Used Methodology:***

Numerical columns including open, close, high, low and volume were selected for scaling. A StandardScaler was applied to standardize these features by subtracting the mean and dividing by the standard deviation. This transformation ensured that all variables contribute equally to the model. The scaling process was applied after data cleaning and feature engineering, and the transformed dataset was verified by displaying sample rows of the normalized data.

In [11]:
# Select numerical columns that need scaling to ensure consistent input ranges for modeling
num_cols = ['open', 'close', 'high', 'low', 'volume']

# Initialize StandardScaler to standardize features by removing mean and scaling to unit variance
scaler = StandardScaler()

# Apply standardization to numerical columns so each feature has comparable scale
cleaned_data[num_cols] = scaler.fit_transform(cleaned_data[num_cols])

# Display first few rows of the normalized dataset to verify scaling
print('Few rows of normalized dataset:')
print(cleaned_data.head())

Few rows of normalized dataset:
           ticker      open     close  adj_close       low      high  \
date                                                                   
2013-05-08    AHH -0.394696 -0.388966   8.493155 -0.399017 -0.395093   
2013-05-09    AHH -0.383169 -0.391129   8.471151 -0.380773 -0.396516   
2013-05-10    AHH -0.391094 -0.387523   8.507822 -0.380773 -0.400787   
2013-05-13    AHH -0.385330 -0.383917   8.544494 -0.377125 -0.397228   
2013-05-14    AHH -0.387492 -0.392572   8.456484 -0.380773 -0.400787   

              volume  price_decade   sector  close_ma7  close_ma30  \
date                                                                 
2013-05-08  2.056068        2010.0  FINANCE        NaN         NaN   
2013-05-09 -0.140739        2010.0  FINANCE        NaN         NaN   
2013-05-10 -0.137725        2010.0  FINANCE        NaN         NaN   
2013-05-13 -0.438439        2010.0  FINANCE        NaN         NaN   
2013-05-14 -0.353348        2010.0  FINANCE

## Encoding Categorical Variables:

***Purpose:***

The purpose of encoding categorical variables was to convert non-numeric data into a numerical format suitable for machine learning models. Since algorithms cannot directly process text-based categories such as sector and ticker, encoding ensures that these variables can be effectively included in predictive analysis.

***Justification on Techniques:***

Different encoding techniques were applied based on the nature of each categorical variable. The sector column was encoded using one-hot encoding because it is a nominal variable with no inherent order and this method prevents the model from assuming any ranking between categories while avoiding multicollinearity using drop_first=True. The ticker column contains a large number of unique stock symbols, so label encoding was used to reduce dimensionality while efficiently converting each ticker into a unique numeric identifier.

***Used Methodology:***

Categorical encoding was performed in two steps.
- First, one-hot encoding was applied to the sector column using pd.get_dummies(), which created separate binary columns for each sector category while dropping the first category to avoid multicollinearity.
- Second, label encoding was applied to the ticker column using LabelEncoder, which assigned a unique integer value to each stock symbol.

These transformations ensured that all categorical variables were converted into a machine-readable format suitable for further modeling and analysis.

In [12]:
# Apply one-hot encoding to the 'sector' categorical variable, which converts each sector into separate binary columns while avoiding multicollinearity using drop_first=True
cleaned_data = pd.get_dummies(cleaned_data, columns=['sector'], drop_first=True)

# Initialize LabelEncoder to convert categorical ticker values into numeric format
le = LabelEncoder()

# Apply label encoding to 'ticker' column to assign a unique numeric value to each stock symbol
cleaned_data['ticker'] = le.fit_transform(cleaned_data['ticker'])

In [13]:
#Drop null values and display first few rows of cleaned dataset
cleaned_data.dropna(inplace=True)
print('Few rows of cleaned dataset:')
print(cleaned_data.head())

Few rows of cleaned dataset:
            ticker      open     close  adj_close       low      high  \
date                                                                    
2013-06-19       6 -0.396136 -0.409159   8.287795 -0.401206 -0.409329   
2013-06-20       6 -0.412705 -0.421419   8.163107 -0.407774 -0.422141   
2013-06-21       6 -0.418468 -0.411323   8.265792 -0.408504 -0.424277   
2013-06-24       6 -0.416307 -0.414929   8.229120 -0.439153 -0.422853   
2013-06-25       6 -0.410544 -0.403389   8.346467 -0.418720 -0.415023   

              volume  price_decade  close_ma7  close_ma30  ...  \
date                                                       ...   
2013-06-19 -0.594012        2010.0  11.478571   11.576000  ...   
2013-06-20 -0.309296        2010.0  11.417143   11.561000  ...   
2013-06-21 -0.112685        2010.0  11.385714   11.551667  ...   
2013-06-24 -0.428933        2010.0  11.347143   11.539000  ...   
2013-06-25 -0.558538        2010.0  11.337143   11.530000  ... 

# Task-3: Integration and Formatting for Modeling

***Purpose:***

The purpose of this step was to prepare a fully structured and analysis-ready dataset for predictive modeling. This involved consolidating all preprocessing steps into a single dataset, splitting the data into training, validation and test sets, and saving the processed datasets for future use. This ensures that the model can be trained, tuned and evaluated in a systematic and reproducible manner.

***Justification:***

Time-series stock data requires careful handling to avoid data leakage and maintain chronological order. Therefore, the dataset was sorted by date to preserve temporal structure before splitting. A 70/15/15 split was used to provide sufficient data for training while also reserving separate datasets for validation and testing. This is important because the training set is used to learn patterns, the validation set is used to tune model performance and prevent overfitting, and the test set provides an unbiased evaluation of final model accuracy. Saving the datasets ensures reproducibility and allows the modeling process to be reused without repeating preprocessing steps.

***Methodology:***

- First, the cleaned dataset was copied into a final consolidated dataset to ensure all preprocessing steps were retained. The data was then sorted in chronological order to maintain the time-series structure.
- Next, the dataset was split sequentially into training (70%), validation (15%) and test (15%) sets without shuffling to prevent future data leakage. - Finally, all datasets were saved as separate CSV files, including the full cleaned dataset and each split. This structured approach ensures proper model validation, reliable testing and consistent evaluation of predictive performance.

**Understanding of the importance of model validation and testing:**

Model validation and testing are essential to ensure that predictive models generalize well to unseen data. The validation set helps in tuning model parameters and detecting overfitting during development, while the test set provides a final unbiased evaluation of model performance. Without proper validation and testing, a model may perform well on training data but fail in real-world scenarios due to overfitting or lack of generalization.


In [14]:
# Create final consolidated dataset after all preprocessing steps
final_data = cleaned_data.copy()#copy of cleaned_data to final_data

# Sort data in chronological order to maintain time-series order
final_data = final_data.sort_index()

# Verify dataset integrity by checking its shape and few rows of it
print("Final dataset shape:", final_data.shape)#display shape of final data
print(final_data.head())#display few rows of final data

Final dataset shape: (465300, 24)
            ticker      open     close  adj_close       low      high  \
date                                                                    
1972-07-13     110 -1.155981 -1.158378   0.000666 -1.153852 -1.160129   
1972-07-14     110 -1.157482 -1.156875   0.000681 -1.153662 -1.159759   
1972-07-17     110 -1.155981 -1.157439   0.000675 -1.153662 -1.159759   
1972-07-18     110 -1.156732 -1.158190   0.000667 -1.155182 -1.160871   
1972-07-19     110 -1.157294 -1.157814   0.000671 -1.153852 -1.160500   

              volume  price_decade  close_ma7  close_ma30  ...  \
date                                                       ...   
1972-07-13  1.668178        1970.0   0.924479    0.885069  ...   
1972-07-14  2.056068        1970.0   0.928943    0.887500  ...   
1972-07-17  1.022699        1970.0   0.929688    0.890017  ...   
1972-07-18  2.056068        1970.0   0.927083    0.892274  ...   
1972-07-19  2.056068        1970.0   0.924851    0.894097 

In [15]:
# Define training set size as 70% of the total dataset
train_size = int(len(final_data) * 0.7)

# Define validation set size as 15% of the total dataset
val_size = int(len(final_data) * 0.15)

# Split dataset sequentially without shuffling to maintain temporal order
train_data = final_data.iloc[:train_size]  # Training data (first 70%)

val_data = final_data.iloc[train_size:train_size + val_size]  # Validation data (next 15%)

test_data = final_data.iloc[train_size + val_size:]  # Test data (remaining 15%)

# Display the shape of each dataset to verify correct splitting
print("Train shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Test shape:", test_data.shape)

Train shape: (325710, 24)
Validation shape: (69795, 24)
Test shape: (69795, 24)


In [16]:
# Save final consolidated dataset
final_data.to_csv("cleaned_stock_data.csv", index=True)

# Save training, validation and test sets
train_data.to_csv("train_data.csv", index=True)
val_data.to_csv("validation_data.csv", index=True)
test_data.to_csv("test_data.csv", index=True)

print("All datasets saved successfully.")

All datasets saved successfully.
